# SupplyMind AI — Feature Engineering

Transform raw, prediction-time-safe SynDelay fields into stable model features.

In [1]:
# -------------------
# Imports
# -------------------

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from supplymind.features.predictions.domain.constants import (
    CATEGORICAL_FEATURES,
    NUMERICAL_FEATURES,
    TARGET_COLUMN,
)
from supplymind.features.predictions.ml.workflow import load_clean_syndelay

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [2]:
# -------------------
# Project configuration
# -------------------

DATASET_PATH = Path("../data/raw/syndelay/syndelay_v1.csv")
REPORT_ROOT = Path("../reports")

assert DATASET_PATH.exists(), (
    f"Dataset not found: {DATASET_PATH}"
)

In [3]:
from supplymind.features.predictions.ml.features import (
    engineer_features,
    build_feature_matrix,
    feature_groups,
    columns_excluded_from_model,
)

df = load_clean_syndelay(DATASET_PATH)
featured = engineer_features(df)

## 1. Date feature creation

In [4]:
temporal_columns = [
    "order_date",
    "order_year",
    "order_month",
    "order_quarter",
    "order_week",
    "order_day",
    "order_weekday",
    "order_hour",
    "order_is_weekend",
]

featured[temporal_columns].head(20)

,order_date,order_year,order_month,order_quarter,order_week,order_day,order_weekday,order_hour,order_is_weekend
0,2015-06-19 12:00:00.000000000,2015,6,2,25,19,4,12,0
1,2016-05-08 09:21:35.999999948,2016,5,2,18,8,6,9,1
2,2017-08-02 06:23:02.400000285,2017,8,3,31,2,2,6,0
3,2015-06-20 21:36:00.000000129,2015,6,2,25,20,5,21,1
4,2016-09-17 19:40:47.999999974,2016,9,3,37,17,5,19,1
5,2017-05-15 21:11:31.200000138,2017,5,2,20,15,0,21,0
6,2015-04-03 18:43:11.999999896,2015,4,2,14,3,4,18,0
7,2017-09-27 01:26:23.999999801,2017,9,3,39,27,2,1,0
8,2015-12-09 12:11:31.200000138,2015,12,4,50,9,2,12,0
9,2015-05-04 09:28:47.999999723,2015,5,2,19,4,0,9,0


### Conclusion

The order timestamp now gives me year, month, quarter, week, day, weekday, hour and a
weekend flag without touching future shipment information. These features are safe because
they are known when the order exists; I deliberately do not derive duration from
`shipping_date`.

## 2. Frequency encoding

In [5]:
frequency_columns = [
    "customer_city_frequency",
    "order_city_frequency",
    "order_state_frequency",
]

featured[[
    "customer_city",
    "order_city",
    "order_state",
    *frequency_columns,
]].head(20)

,customer_city,order_city,order_state,customer_city_frequency,order_city_frequency,order_state_frequency
0,Caguas,Fort-de-France,Martinique,0.372968,0.001325,0.001325
1,Caguas,Luanshya,Copperbelt,0.372968,0.000090,0.000695
2,Caguas,Ankara,Ankara,0.372968,0.002264,0.002952
3,Caguas,Tegucigalpa,Francisco Morazan,0.372968,0.011371,0.011371
4,Madison,Leon,Leon,0.000302,0.003962,0.003962
5,Caguas,Sari,Mazandaran,0.372968,0.000187,0.000341
6,Florissant,Houston,Texas,0.002135,0.004798,0.012464
7,Pompano Beach,Wollongong,New South Wales,0.003299,0.002071,0.008393
8,Elmhurst,Nuneaton,England,0.001254,0.000174,0.036582
9,Mount Pleasant,Paris,Isle of France,0.001383,0.005672,0.030324


### Conclusion

The frequency features reduce `customer_city` (563 values), `order_city` (3,505) and
`order_state` (1,054) to three numeric inputs instead of thousands of dummy columns.
For production, these mappings are learned from the training set inside
`ShipmentFeatureEngineer` and saved with the model.

## 3. Explicit model exclusions

In [6]:
columns_excluded_from_model()

['category_id',
 'customer_id',
 'customer_zipcode',
 'delivery_outcome',
 'department_id',
 'is_delayed',
 'label',
 'order_customer_id',
 'order_id',
 'order_item_cardprod_id',
 'order_item_id',
 'order_status',
 'product_card_id',
 'product_category_id',
 'shipping_date']

### Conclusion

The excluded set is intentional: target columns, IDs, redundant category IDs,
`customer_zipcode`, `shipping_date` and ambiguous `order_status` do not enter the model.
This keeps the model focused on information that can plausibly exist at prediction time
instead of memorizing records or leaking the outcome.

## 4. Final feature groups

In [7]:
numerical_features, categorical_features = feature_groups()

print("Numerical features:", len(numerical_features))
display(pd.Series(numerical_features, name="numerical_feature").to_frame())

print("Categorical features:", len(categorical_features))
display(pd.Series(categorical_features, name="categorical_feature").to_frame())

Numerical features: 24


,numerical_feature
0,profit_per_order
1,sales_per_customer
2,latitude
3,longitude
4,order_item_discount
5,order_item_discount_rate
6,order_item_product_price
7,order_item_profit_ratio
8,order_item_quantity
9,sales


Categorical features: 11


,categorical_feature
0,payment_type
1,category_name
2,customer_country
3,customer_segment
4,customer_state
5,department_name
6,market
7,order_country
8,order_region
9,product_name


## 5. Final feature matrix

In [8]:
X = build_feature_matrix(featured)
y = featured[TARGET_COLUMN]

print("X:", X.shape)
print("y:", y.shape)
display(X.head())

X: (155488, 35)
y: (155488,)


,profit_per_order,sales_per_customer,latitude,longitude,order_item_discount,order_item_discount_rate,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total_amount,order_profit_per_order,product_price,order_year,order_month,order_quarter,order_week,order_day,order_weekday,order_hour,order_is_weekend,customer_city_frequency,order_city_frequency,order_state_frequency,payment_type,category_name,customer_country,customer_segment,customer_state,department_name,market,order_country,order_region,product_name,shipping_mode
0,-32.924488,278.95000,18.247585,-66.370610,77.9940,0.20,129.99,-0.22,3.0,389.97,311.9760,-68.634720,129.99,2015,6,2,25,19,4,12,0,0.372968,0.001325,0.001325,PAYMENT,Kids' Golf Clubs,Puerto Rico,Corporate,PR,Outdoors,LATAM,Martinica,Caribbean,GolfBuddy VT3 GPS Watch,Second Class
1,107.874500,263.98000,18.228312,-66.370510,35.9940,0.12,59.99,0.36,5.0,299.95,263.9560,95.024160,59.99,2016,5,2,18,8,6,9,1,0.372968,0.000090,0.000695,DEBIT,Cleats,Puerto Rico,Corporate,PR,Apparel,Africa,Zambia,East Africa,Perfect Fitness Perfect Rip Deck,Same Day
2,35.770718,109.65013,18.287943,-66.370514,10.7982,0.09,59.99,0.33,2.0,119.98,109.1818,36.029994,59.99,2017,8,3,31,2,2,6,0,0.372968,0.002264,0.002952,PAYMENT,Cleats,Puerto Rico,Consumer,PR,Apparel,Pacific Asia,Turkey,West Asia,Perfect Fitness Perfect Rip Deck,Standard Class
3,43.587560,113.09000,18.234800,-66.370600,19.4985,0.15,129.99,0.35,1.0,129.99,110.4915,38.672025,129.99,2015,6,2,25,20,5,21,1,0.372968,0.011371,0.011371,PAYMENT,Men's Footwear,Puerto Rico,Consumer,PR,Apparel,LATAM,Honduras,Central America,Nike Men's CJ Elite 2 TD Football Cleat,Second Class
4,49.804802,191.98090,33.920456,-89.977650,3.9996,0.02,99.99,0.29,2.0,199.98,195.9804,56.834316,99.99,2016,9,3,37,17,5,19,1,0.000302,0.003962,0.003962,PAYMENT,Cardio Equipment,EE. UU.,Corporate,WI,Footwear,LATAM,Nicaragua,Central America,Nike Men's Free 5.0+ Running Shoe,Standard Class
